In [0]:
%sql
-- ==========================================================================
-- GOLD LAYER DATA VALIDATION SUITE
-- ==========================================================================
-- Comprehensive validation checks for dimension and fact tables
-- Tests: row counts, PK uniqueness, FK integrity, NULLs, data quality

-- ----------------------------------------------------------
-- 1. ROW COUNT SUMMARY
-- ----------------------------------------------------------
SELECT 'Row Count Summary' AS validation_check, '' AS details, '' AS status, 0 AS issue_count
UNION ALL
SELECT '  DimCourse', CAST(COUNT(*) AS STRING), 'INFO', COUNT(*) FROM DimCourse
UNION ALL
SELECT '  DimDate', CAST(COUNT(*) AS STRING), 'INFO', COUNT(*) FROM DimDate
UNION ALL
SELECT '  DimStudent', CAST(COUNT(*) AS STRING), 'INFO', COUNT(*) FROM DimStudent
UNION ALL
SELECT '  DimModulePresentation', CAST(COUNT(*) AS STRING), 'INFO', COUNT(*) FROM DimModulePresentation
UNION ALL
SELECT '  DimDemographics', CAST(COUNT(*) AS STRING), 'INFO', COUNT(*) FROM DimDemographics
UNION ALL
SELECT '  FactVLEInteractions', CAST(COUNT(*) AS STRING), 'INFO', COUNT(*) FROM FactVLEInteractions
UNION ALL
SELECT '  FactAssessments', CAST(COUNT(*) AS STRING), 'INFO', COUNT(*) FROM FactAssessments

UNION ALL SELECT '', '', '', 0
UNION ALL

-- ----------------------------------------------------------
-- 2. PRIMARY KEY UNIQUENESS CHECKS
-- ----------------------------------------------------------
SELECT 'Primary Key Uniqueness' AS validation_check, '' AS details, '' AS status, 0 AS issue_count
UNION ALL
SELECT '  DimCourse PK',
       CONCAT('Duplicates: ', CAST(dup_count AS STRING)),
       CASE WHEN dup_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       dup_count
FROM (
    SELECT COUNT(*) - COUNT(DISTINCT code_module) AS dup_count FROM DimCourse
)
UNION ALL
SELECT '  DimDate PK',
       CONCAT('Duplicates: ', CAST(dup_count AS STRING)),
       CASE WHEN dup_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       dup_count
FROM (
    SELECT COUNT(*) - COUNT(DISTINCT date) AS dup_count FROM DimDate
)
UNION ALL
SELECT '  DimStudent PK',
       CONCAT('Duplicates: ', CAST(dup_count AS STRING)),
       CASE WHEN dup_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       dup_count
FROM (
    SELECT COUNT(*) - COUNT(DISTINCT id_student) AS dup_count FROM DimStudent
)
UNION ALL
SELECT '  DimModulePresentation PK',
       CONCAT('Duplicates: ', CAST(dup_count AS STRING)),
       CASE WHEN dup_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       dup_count
FROM (
    SELECT COUNT(*) - COUNT(DISTINCT code_module, code_presentation) AS dup_count FROM DimModulePresentation
)
UNION ALL
SELECT '  DimDemographics PK',
       CONCAT('Duplicates: ', CAST(dup_count AS STRING)),
       CASE WHEN dup_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       dup_count
FROM (
    SELECT COUNT(*) - COUNT(DISTINCT id_student, code_module, code_presentation) AS dup_count FROM DimDemographics
)
UNION ALL
SELECT '  FactVLEInteractions PK',
       CONCAT('Duplicates: ', CAST(dup_count AS STRING)),
       CASE WHEN dup_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       dup_count
FROM (
    SELECT COUNT(*) - COUNT(DISTINCT id_student, code_module, code_presentation, date, id_site) AS dup_count FROM FactVLEInteractions
)
UNION ALL
SELECT '  FactAssessments PK',
       CONCAT('Duplicates: ', CAST(dup_count AS STRING)),
       CASE WHEN dup_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       dup_count
FROM (
    SELECT COUNT(*) - COUNT(DISTINCT id_student, code_module, code_presentation, date, id_assessment) AS dup_count FROM FactAssessments
)

UNION ALL SELECT '', '', '', 0
UNION ALL

-- ----------------------------------------------------------
-- 3. NULL CHECKS ON REQUIRED COLUMNS
-- ----------------------------------------------------------
SELECT 'NULL Checks (Required Columns)' AS validation_check, '' AS details, '' AS status, 0 AS issue_count
UNION ALL
SELECT '  DimCourse.code_module',
       CONCAT('NULLs: ', CAST(null_count AS STRING)),
       CASE WHEN null_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       null_count
FROM (
    SELECT COUNT(*) - COUNT(code_module) AS null_count FROM DimCourse
)
UNION ALL
SELECT '  DimDate.date',
       CONCAT('NULLs: ', CAST(null_count AS STRING)),
       CASE WHEN null_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       null_count
FROM (
    SELECT COUNT(*) - COUNT(date) AS null_count FROM DimDate
)
UNION ALL
SELECT '  DimStudent.id_student',
       CONCAT('NULLs: ', CAST(null_count AS STRING)),
       CASE WHEN null_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       null_count
FROM (
    SELECT COUNT(*) - COUNT(id_student) AS null_count FROM DimStudent
)
UNION ALL
SELECT '  FactVLEInteractions FKs',
       CONCAT('NULLs in id_student/code_module/code_presentation/date/id_site: ', CAST(null_count AS STRING)),
       CASE WHEN null_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       null_count
FROM (
    SELECT SUM(CASE WHEN id_student IS NULL OR code_module IS NULL OR code_presentation IS NULL OR date IS NULL OR id_site IS NULL THEN 1 ELSE 0 END) AS null_count
    FROM FactVLEInteractions
)
UNION ALL
SELECT '  FactAssessments FKs',
       CONCAT('NULLs in id_student/id_assessment/date: ', CAST(null_count AS STRING)),
       CASE WHEN null_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       null_count
FROM (
    SELECT SUM(CASE WHEN id_student IS NULL OR id_assessment IS NULL OR date IS NULL THEN 1 ELSE 0 END) AS null_count
    FROM FactAssessments
)

UNION ALL SELECT '', '', '', 0
UNION ALL

-- ----------------------------------------------------------
-- 4. FOREIGN KEY INTEGRITY CHECKS
-- ----------------------------------------------------------
SELECT 'Foreign Key Integrity' AS validation_check, '' AS details, '' AS status, 0 AS issue_count
UNION ALL
SELECT '  DimModulePresentation -> DimCourse',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM DimModulePresentation mp
    LEFT JOIN DimCourse c ON mp.code_module = c.code_module
    WHERE c.code_module IS NULL
)
UNION ALL
SELECT '  DimDemographics -> DimStudent',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM DimDemographics d
    LEFT JOIN DimStudent s ON d.id_student = s.id_student
    WHERE s.id_student IS NULL
)
UNION ALL
SELECT '  DimDemographics -> DimModulePresentation',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM DimDemographics d
    LEFT JOIN DimModulePresentation mp
        ON d.code_module = mp.code_module
        AND d.code_presentation = mp.code_presentation
    WHERE mp.code_module IS NULL
)
UNION ALL
SELECT '  FactVLEInteractions -> DimStudent',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM FactVLEInteractions f
    LEFT JOIN DimStudent s ON f.id_student = s.id_student
    WHERE s.id_student IS NULL
)
UNION ALL
SELECT '  FactVLEInteractions -> DimModulePresentation',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM FactVLEInteractions f
    LEFT JOIN DimModulePresentation mp
        ON f.code_module = mp.code_module
        AND f.code_presentation = mp.code_presentation
    WHERE mp.code_module IS NULL
)
UNION ALL
SELECT '  FactVLEInteractions -> DimDate',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM FactVLEInteractions f
    LEFT JOIN DimDate d ON f.date = d.date
    WHERE d.date IS NULL
)
UNION ALL
SELECT '  FactAssessments -> DimStudent',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM FactAssessments f
    LEFT JOIN DimStudent s ON f.id_student = s.id_student
    WHERE s.id_student IS NULL
)
UNION ALL
SELECT '  FactAssessments -> DimModulePresentation',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM FactAssessments f
    LEFT JOIN DimModulePresentation mp
        ON f.code_module = mp.code_module
        AND f.code_presentation = mp.code_presentation
    WHERE mp.code_module IS NULL
)
UNION ALL
SELECT '  FactAssessments -> DimDate',
       CONCAT('Orphaned records: ', CAST(orphan_count AS STRING)),
       CASE WHEN orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       orphan_count
FROM (
    SELECT COUNT(*) AS orphan_count
    FROM FactAssessments f
    LEFT JOIN DimDate d ON f.date = d.date
    WHERE d.date IS NULL
)

UNION ALL SELECT '', '', '', 0
UNION ALL

-- ----------------------------------------------------------
-- 5. DATA QUALITY CHECKS
-- ----------------------------------------------------------
SELECT 'Data Quality Checks' AS validation_check, '' AS details, '' AS status, 0 AS issue_count
UNION ALL
SELECT '  DimDate: relative_week consistency',
       CONCAT('Mismatches: ', CAST(mismatch_count AS STRING)),
       CASE WHEN mismatch_count = 0 THEN 'PASS' ELSE 'FAIL' END,
       mismatch_count
FROM (
    SELECT COUNT(*) AS mismatch_count
    FROM DimDate
    WHERE relative_week != FLOOR(date / 7)
)
UNION ALL
SELECT '  FactVLEInteractions: negative clicks',
       CONCAT('Negative values: ', CAST(neg_count AS STRING)),
       CASE WHEN neg_count = 0 THEN 'PASS' ELSE 'WARN' END,
       neg_count
FROM (
    SELECT COUNT(*) AS neg_count
    FROM FactVLEInteractions
    WHERE sum_click < 0
)
UNION ALL
SELECT '  FactAssessments: score range (0-100)',
       CONCAT('Out of range: ', CAST(out_range AS STRING)),
       CASE WHEN out_range = 0 THEN 'PASS' ELSE 'FAIL' END,
       out_range
FROM (
    SELECT COUNT(*) AS out_range
    FROM FactAssessments
    WHERE score IS NOT NULL AND (score < 0 OR score > 100)
)
UNION ALL
SELECT '  FactAssessments: weight range (0-100)',
       CONCAT('Out of range: ', CAST(out_range AS STRING)),
       CASE WHEN out_range = 0 THEN 'PASS' ELSE 'WARN' END,
       out_range
FROM (
    SELECT COUNT(*) AS out_range
    FROM FactAssessments
    WHERE weight IS NOT NULL AND (weight < 0 OR weight > 100)
)
UNION ALL
SELECT '  DimStudent: withdrawn flag consistency',
       CONCAT('Inconsistent: ', CAST(incon_count AS STRING), ' (withdrawn=true but final_result not Withdrawn)'),
       CASE WHEN incon_count = 0 THEN 'PASS' ELSE 'WARN' END,
       incon_count
FROM (
    SELECT COUNT(*) AS incon_count
    FROM DimStudent
    WHERE is_withdrawn = true AND final_result != 'Withdrawn'
)

UNION ALL SELECT '', '', '', 0
UNION ALL

-- ----------------------------------------------------------
-- 6. VALIDATION SUMMARY
-- ----------------------------------------------------------
SELECT 'VALIDATION SUMMARY' AS validation_check,
       CONCAT('Total Issues: ', CAST(SUM(issue_count) AS STRING)) AS details,
       CASE WHEN SUM(issue_count) = 0 THEN 'ALL PASS' ELSE 'REVIEW NEEDED' END AS status,
       SUM(issue_count) AS issue_count
FROM (
    -- Collect all issue counts from above (simplified - in production you'd capture all)
    SELECT 0 AS issue_count  -- Placeholder for aggregation
)

ORDER BY validation_check;